# **RAG Pipeline : Data Ingestion to Vector DB Pipeline**

In [3]:
import os
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader, DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [8]:
def process_pdf_files(pdf_directory):
    all_docs = []
    pdf_dir = Path(pdf_directory)
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    print(f"Found {len(pdf_files)} PDF files in {pdf_directory}")
    for pdf_file in pdf_files:
        print(f"\nprocessing file: {pdf_file}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            for doc in documents:
                doc.metadata['source-file'] = pdf_file.name
                doc.metadata['type'] = 'pdf'
            all_docs.extend(documents)
            print(f"Loaded {len(documents)} pages from {pdf_file.name}")
        except Exception as e:
            print(f"Error processing {pdf_file.name}: {e}")
        
    print(f"\nTotal documents loaded: {len(all_docs)}")
    return all_docs

pdf_directory = "../data/pdf_files"
documents = process_pdf_files(pdf_directory)

Found 1 PDF files in ../data/pdf_files

processing file: ..\data\pdf_files\osqb.pdf
Loaded 6 pages from osqb.pdf

Total documents loaded: 6


In [14]:
### Text splitting get into chunks
def text_splitter(documents,chunk_size = 1000, chunk_overlap=200):
    text_splt = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, 
        chunk_overlap=chunk_overlap,
        length_function = len,
        separators=["\n\n", "\n", " ", ""]
    )
    chunks = text_splt.split_documents(documents)
    print(f"Split {len(documents)} document(s) into {len(chunks)} chunks.")    
    if chunks:
        print(f"\nExample chunk:")
        print(f"Content: {chunks[0].page_content[:200]}...")
        print(f"Metadata: {chunks[0].metadata}")
    
    return chunks   

chunks = text_splitter(documents)

Split 6 document(s) into 15 chunks.

Example chunk:
Content: Operating Systems — Unit-wise & Topic-wise Repeated Question Bank Operating Systems — Unit-wise & Topic-wise Repeated Question Bank
UNIT 1 — INTRODUCTION & PROCESS MANAGEMENT
UNIT 1 — INTRODUCTION & P...
Metadata: {'producer': 'Qt 4.8.7', 'creator': 'wkhtmltopdf 0.12.4', 'creationdate': '2026-05-16T17:04:14+00:00', 'title': 'Markdown To PDF', 'source': '..\\data\\pdf_files\\osqb.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1', 'source-file': 'osqb.pdf', 'type': 'pdf'}


### **Embedding and VectorStore DB**

In [15]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

d:\gitfolder\rag basics\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [16]:
class EmbeddingManager:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model = SentenceTransformer(model_name)

        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        try:
            print(f"Loading embedding model: {self.model_name}...")
            self.model = SentenceTransformer(self.model_name)
            print("Model loaded successfully. Embedding Dimension:", {self.model.get_sentence_embedding_dimension()})
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise
    
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        if not self.model:
            raise ValueError("Embedding model is not loaded.")
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar = True)
        print("Embeddings generated successfully with shape:", {embeddings.shape})
        return embeddings
    

embedding_manager = EmbeddingManager()
embedding_manager

d:\gitfolder\rag basics\venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\hrida\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 851.94it/s]


Loading embedding model: all-MiniLM-L6-v2...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1429.37it/s]


Model loaded successfully. Embedding Dimension: {384}


C:\Users\hrida\AppData\Local\Temp\ipykernel_23720\2859781826.py:13: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Model loaded successfully. Embedding Dimension:", {self.model.get_sentence_embedding_dimension()})


### **VectorStore**

In [ ]:
class VectorStore:
    def __init__(self, collection_name : str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path = self.persist_directory)

            self.collection = self.client.get_or_create_collection(name = self.collection_name, metadata = "Description : PDF document chunks with embeddings")
            print(f"Vector store initialized successfully with collection: {self.collection_name}")
            print(f"Existing collections: {self.collection.count()}")
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents:List[Any], embeddings: np.ndarray):
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents and embeddings must match.")
        print(f"Adding {len(documents)} documents to vector store.")

        #prepare data for embeddings
        ids = []
        metadata = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embeddings) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            #prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadata.append(metadata)

            documents_text.append(doc.page_content)
            embeddings_list.append(embeddings.tolist())

        try:
            self.collection.add(
                ids = ids,
                documents = documents_text,
                metadatas = metadata,
                embeddings = embeddings_list
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise


vectorstore=VectorStore()
vectorstore
                
